In [1]:
# imports
import MDAnalysis as mda
import pandas as pd
import nglview as nv
from tqdm.notebook import tqdm
from merge_structures import assign_segment_ids, identify_subaggregates, merge_structures


In [2]:
# load the structure files, assign unique segment IDs for every protein, identify aggregates in the structures
# and save relevant paramenters in a pandas dataframe
existing_segids = {}
protein_data = {}
for prot in tqdm(["hpl2", "lin13", "lin65", "met2"], desc="Loading structure files", leave=False, unit="Files"):
    x = mda.Universe(f"input_structures/single_comp_structures/{prot}.pdb")
    u = x.copy()

    # Add unique segment IDs to differentiate between protein chains
    u = assign_segment_ids(u, prot, existing_segids)
    
    # assign known universe dimensions
    u.dimensions = [600,600,3000,90,90,90]

    # Analyze chains and clusters
    clusters = identify_subaggregates(u)    
    protein_data[prot] = {
        'universe': u,
        "n_atoms": len(u.atoms),
        'subaggregates': clusters,
        'rg_max': max([prot.atoms.radius_of_gyration() for prot in u.segments]),
        "n_proteins": len(u.segments), 
        "box_dimensions": u.dimensions
    }

proteins_df = pd.DataFrame.from_dict(protein_data, orient='index')
proteins_df

Loading structure files:   0%|          | 0/4 [00:00<?, ?Files/s]

,universe,n_atoms,subaggregates,rg_max,n_proteins,box_dimensions
hpl2,<Universe with 14000 atoms>,14000,"[(<Atom 1: CA of type C of resname MET, resid ...",58.676363,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
lin13,<Universe with 89920 atoms>,89920,"[(<Atom 1: CA of type C of resname MET, resid ...",139.430734,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
lin65,<Universe with 29120 atoms>,29120,"[(<Atom 1: CA of type C of resname MET, resid ...",186.969647,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"
met2,<Universe with 14000 atoms>,14000,"[(<Atom 1: CA of type C of resname MET, resid ...",58.676363,40,"[600.0, 600.0, 3000.0, 90.0, 90.0, 90.0]"


In [3]:
merged = merge_structures(proteins_df)

Initialized the structure with lin13 (1 aggregates).
New structure dimensions set at x: 709.9295739719539, y: 709.9295739719539, z: 2000.


placing structures:   0%|          | 0/3 [00:00<?, ?struct/s]

explore structure fit:   0%|          | 0/3 [00:00<?, ?struct/s]

scan z offsets:   0%|          | 0/14 [00:00<?, ?it/s]

scan z offsets:   0%|          | 0/14 [00:00<?, ?it/s]

...

scan z offsets:   0%|          | 0/14 [00:00<?, ?it/s]

Placed hpl2: 15/31 clusters fit around Z=1380.0 (Score: 0.016)


explore structure fit:   0%|          | 0/1 [00:00<?, ?struct/s]

scan z offsets:   0%|          | 0/14 [00:00<?, ?it/s]

Placed met2: 12/31 clusters fit around Z=250.0 (Score: 0.013)


In [4]:
view = nv.show_mdanalysis(merged.atoms)

# add the unit cell (the box)
view.add_unitcell()

view

c:\Users\phili\anaconda3\envs\MD_b_practical\Lib\site-packages\MDAnalysis\coordinates\PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


NGLWidget()

In [5]:
from MDAnalysis.lib.distances import capped_distance

def check_true_structure_overlaps(u, cutoff=4.0):
    """
    Groups segments by their 2-character prefix and checks for 
    collisions between these groups (ignoring internal chain-chain contacts).
    """
    # 1. Group segments by their prefix (first 2 chars)
    # Result: {'H2': AtomGroup, 'L1': AtomGroup, ...}
    prefixes = sorted(list(set(s.segid[:2] for s in u.segments)))
    structure_groups = {p: u.select_atoms(f"segid {p}*") for p in prefixes}
    
    total_collisions = 0
    checked_pairs = []

    # 2. Nested loop to check only DIFFERENT structures
    for i, pref1 in enumerate(prefixes):
        for j, pref2 in enumerate(prefixes):
            if i >= j: continue # Avoid double-counting and self-interaction
            
            group1 = structure_groups[pref1]
            group2 = structure_groups[pref2]
            
            # Use capped_distance for the two distinct AtomGroups
            pairs = capped_distance(group1.positions, 
                                    group2.positions, 
                                    max_cutoff=cutoff, 
                                    box=u.dimensions)
            
            count = len(pairs[0])
            if count > 0:
                print(f"Collision: {pref1} <-> {pref2} | {count} atoms overlapping")
            
            total_collisions += count
            
    return total_collisions

# Run the check
true_overlaps = check_true_structure_overlaps(merged)
print(f"--- Final Report: {true_overlaps} true inter-structure collisions found ---")

Collision: H2 <-> M2 | 41 atoms overlapping
Collision: L6 <-> M2 | 14 atoms overlapping
--- Final Report: 55 true inter-structure collisions found ---
